# Part C: Mage Iris ML Pipeline
**Course:** Data Engineering - Pipeline Orchestration  
**Student Name:** Golla Teja  
**Roll Number:** 23BCS091  
**Repository:** https://github.com/teja-0311/freshcart-pipeline  

---
### Pipeline Architecture
```
load_iris (data_loader)
       |
       v
validate_data (transformer - custom quality gate)
       |
       v
prepare_data (transformer - 80/20 train/test split)
       |
       v
train_model (transformer - DecisionTreeClassifier with accuracy gate >= 90%)
      / \
     /   \
    v     v
save_results (data_exporter)   evaluate_model (transformer)
[iris_predictions.csv]          [Confusion Matrix & Classification Metrics]
```


## 1. Setup & Environment Configuration
Load dependencies and ensure repo root is in python path.

In [1]:
import sys, os
from pathlib import Path

# Ensure repository root is on sys.path
cwd = Path(os.getcwd())
repo_root = cwd.parent if cwd.name == "iris_project" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

print("All dependencies loaded successfully.")
print("Python version:", sys.version.split()[0])
print("Repository root:", repo_root.resolve())


All dependencies loaded successfully.
Python version: 3.12.4
Repository root: C:\Users\tejag\Music\DE_Assignment


## 2. Block 1: `load_iris` (data_loader)
Loads the Iris dataset from scikit-learn and runs `@test: test_output`.

In [2]:
from iris_project.data_loaders import load_iris

df_raw = load_iris.load_data()
print("Loaded raw Iris dataset. Shape:", df_raw.shape)
print("\nFirst 5 rows:")
print(df_raw.head().to_string())

# Run block test
load_iris.test_output(df_raw)
print("\n@test: test_output PASSED!")


Loaded raw Iris dataset. Shape: (150, 5)

First 5 rows:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  target
0                5.1               3.5                1.4               0.2       0
1                4.9               3.0                1.4               0.2       0
2                4.7               3.2                1.3               0.2       0
3                4.6               3.1                1.5               0.2       0
4                5.0               3.6                1.4               0.2       0

@test: test_output PASSED!


## 3. Block 2: `validate_data` (custom transformer - Part C4)
Validates dataset integrity: verifies 0 null values and exactly 3 classes.

In [3]:
from iris_project.transformers import validate_data

df_validated = validate_data.transform(df_raw)
print("Data validation passed. Output shape:", df_validated.shape)

# Run block tests
validate_data.test_no_missing_values(df_validated)
print("@test: test_no_missing_values PASSED!")

validate_data.test_exactly_three_classes(df_validated)
print("@test: test_exactly_three_classes PASSED!")


Data validation passed. Output shape: (150, 5)
@test: test_no_missing_values PASSED!
@test: test_exactly_three_classes PASSED!


## 4. Block 3: `prepare_data` (transformer)
Splits data into 80% train (120 rows) and 20% test (30 rows), stratified by target.

In [4]:
from iris_project.transformers import prepare_data

df_prepared = prepare_data.transform(df_validated)
train_count = (df_prepared["split"] == "train").sum()
test_count = (df_prepared["split"] == "test").sum()

print(f"Total rows: {len(df_prepared)}")
print(f"Train split rows: {train_count} (80%)")
print(f"Test split rows:  {test_count} (20%)")

# Run block test
prepare_data.test_split(df_prepared)
print("\n@test: test_split PASSED!")


Total rows: 150
Train split rows: 120 (80%)
Test split rows:  30 (20%)

@test: test_split PASSED!


## 5. Block 4: `train_model` & Accuracy Test Gate (Part C1 & C2)
Trains DecisionTreeClassifier and applies `@test: test_accuracy` gate (threshold >= 90%).

In [5]:
from iris_project.transformers import train_model

model_results = train_model.transform(df_prepared, max_depth=3)
acc = model_results["accuracy"]

print(f"Trained DecisionTreeClassifier(max_depth=3)")
print(f"Test Accuracy: {acc:.4f} ({acc*100:.1f}%)")

# Run test gate
train_model.test_accuracy(model_results)
print("\n@test: test_accuracy PASSED (Accuracy >= 90% threshold)!")


Model trained with max_depth=3. Test Accuracy: 0.9667 (96.7%)
Trained DecisionTreeClassifier(max_depth=3)
Test Accuracy: 0.9667 (96.7%)

@test: test_accuracy PASSED (Accuracy >= 90% threshold)!


## 6. Block 5: `save_results` (data_exporter)
Exports the 30 test predictions to `iris_predictions.csv`.

In [6]:
from iris_project.data_exporters import save_results

save_results.export_data(model_results)

# Run block test
save_results.test_predictions_file_created(model_results)
print("@test: test_predictions_file_created PASSED!")

preds_df = pd.read_csv("iris_predictions.csv")
print(f"\nLoaded {len(preds_df)} saved predictions from iris_predictions.csv:")
print(preds_df.head(10).to_string())


Saved 30 predictions to iris_predictions.csv
@test: test_predictions_file_created PASSED!

Loaded 30 saved predictions from iris_predictions.csv:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  target split  predicted_target
0                5.0               3.4                1.5               0.2       0  test                 0
1                5.4               3.7                1.5               0.2       0  test                 0
2                5.8               4.0                1.2               0.2       0  test                 0
3                5.7               3.8                1.7               0.3       0  test                 0
4                5.4               3.4                1.7               0.2       0  test                 0
5                4.6               3.6                1.0               0.2       0  test                 0
6                5.2               3.4                1.4               0.2       0  test         

## 7. Block 6: `evaluate_model` (Evaluation Branch - Part C3)
Computes Confusion Matrix and Classification Report metrics table.

In [7]:
from iris_project.transformers import evaluate_model

eval_results = evaluate_model.transform(model_results)

print("Confusion Matrix:")
cm_df = pd.DataFrame(eval_results["confusion_matrix"], 
                     index=["Setosa (0)", "Versicolor (1)", "Virginica (2)"],
                     columns=["Pred Setosa", "Pred Versicolor", "Pred Virginica"])
print(cm_df.to_string())

print("\nClassification Report Metrics Table:")
metrics_df = pd.DataFrame(eval_results["metrics_table"]).T if isinstance(eval_results["metrics_table"], dict) else eval_results["metrics_table"]
print(metrics_df.to_string())

evaluate_model.test_evaluation_shape(eval_results)
print("\n@test: test_evaluation_shape PASSED!")


=== Model Evaluation ===
Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Metrics Table:
              precision    recall  f1-score    support
0              1.000000  1.000000  1.000000  10.000000
1              1.000000  0.900000  0.947368  10.000000
2              0.909091  1.000000  0.952381  10.000000
accuracy       0.966667  0.966667  0.966667   0.966667
macro avg      0.969697  0.966667  0.966583  30.000000
weighted avg   0.969697  0.966667  0.966583  30.000000
Confusion Matrix:
                Pred Setosa  Pred Versicolor  Pred Virginica
Setosa (0)               10                0               0
Versicolor (1)            0                9               1
Virginica (2)             0                0              10

Classification Report Metrics Table:
              0          1          2  accuracy  macro avg  weighted avg
precision   1.0   1.000000   0.909091  0.966667   0.969697      0.969697
recall      1.0   0.900000   1.000000  0.966667   0.966667      0.966667


## 8. Part C2: Runtime Parameter Experiments (max_depth = 1..5)
Demonstrates how failing `@test` at `max_depth = 1` blocks file creation, while `max_depth = 2..5` succeed.

In [8]:
results = []

for depth in range(1, 6):
    pred_file = "iris_predictions.csv"
    if os.path.exists(pred_file):
        os.remove(pred_file)
        
    try:
        res = train_model.transform(df_prepared, max_depth=depth)
        train_model.test_accuracy(res)
        gate_passed = True
        save_results.export_data(res)
        csv_created = os.path.exists(pred_file)
    except AssertionError as e:
        gate_passed = False
        csv_created = False
        
    results.append({
        "max_depth": depth,
        "Test Accuracy": f"{res['accuracy']:.4f} ({res['accuracy']*100:.1f}%)",
        "test_accuracy passed?": "PASSED" if gate_passed else "FAILED (blocked)",
        "iris_predictions.csv created?": "YES (30 rows)" if csv_created else "NO (file blocked)"
    })

exp_df = pd.DataFrame(results)
print(exp_df.to_string(index=False))

# Re-run at depth 3 to leave predictions file intact
res = train_model.transform(df_prepared, max_depth=3)
save_results.export_data(res)


Model trained with max_depth=1. Test Accuracy: 0.6667 (66.7%)
Model trained with max_depth=2. Test Accuracy: 0.9333 (93.3%)
Saved 30 predictions to iris_predictions.csv
Model trained with max_depth=3. Test Accuracy: 0.9667 (96.7%)
Saved 30 predictions to iris_predictions.csv
Model trained with max_depth=4. Test Accuracy: 0.9333 (93.3%)
Saved 30 predictions to iris_predictions.csv
Model trained with max_depth=5. Test Accuracy: 0.9333 (93.3%)
Saved 30 predictions to iris_predictions.csv
 max_depth  Test Accuracy test_accuracy passed? iris_predictions.csv created?
         1 0.6667 (66.7%)      FAILED (blocked)             NO (file blocked)
         2 0.9333 (93.3%)                PASSED                 YES (30 rows)
         3 0.9667 (96.7%)                PASSED                 YES (30 rows)
         4 0.9333 (93.3%)                PASSED                 YES (30 rows)
         5 0.9333 (93.3%)                PASSED                 YES (30 rows)
Model trained with max_depth=3. Test Accur

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,split,predicted_target
7,5.0,3.4,1.5,0.2,0,test,0
10,5.4,3.7,1.5,0.2,0,test,0
14,5.8,4.0,1.2,0.2,0,test,0
18,5.7,3.8,1.7,0.3,0,test,0
20,5.4,3.4,1.7,0.2,0,test,0
22,4.6,3.6,1.0,0.2,0,test,0
28,5.2,3.4,1.4,0.2,0,test,0
38,4.4,3.0,1.3,0.2,0,test,0
42,4.4,3.2,1.3,0.2,0,test,0
49,5.0,3.3,1.4,0.2,0,test,0


## 9. Part C4: Custom Block Gate Demonstration
Demonstrates that `validate_data` catches null values and invalid class counts.

In [9]:
# Case 1: Injected null value
df_bad_null = df_raw.copy()
df_bad_null.loc[0, "sepal length (cm)"] = None
try:
    validate_data.transform(df_bad_null)
    print("UNEXPECTED PASS")
except ValueError as e:
    print("SUCCESSFULLY CAUGHT NULL VALUE:", e)

# Case 2: Injected 4th class
df_bad_class = df_raw.copy()
df_bad_class.loc[0, "target"] = 99
try:
    validate_data.transform(df_bad_class)
    print("UNEXPECTED PASS")
except ValueError as e:
    print("SUCCESSFULLY CAUGHT INVALID CLASS COUNT:", e)


SUCCESSFULLY CAUGHT NULL VALUE: Dataset contains unexpected missing/null values
SUCCESSFULLY CAUGHT INVALID CLASS COUNT: Expected exactly 3 target classes, found 4
